Preprocessing by Nguyen

In [ ]:
import ast
import pandas as pd
import numpy as np
import spacy
from typing import Literal
import pickle

class Preprocess:
    def __init__(self):
        pass

Text = tuple[Literal[-1, 1], np.ndarray]
Dialog = list[Text]

def get_embeddings(file: str) -> dict:
    embeddings = {}
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], "float32")
            embeddings[word] = vector
    return embeddings

def preprocess_data(file: str = "Data/train.csv", embeddings_file: str = "vectors/glove.6B.50d.txt") -> list[Dialog]:
    embeddings = get_embeddings(embeddings_file)
    print("Embeddings loaded with size:", len(embeddings))
    nlp = spacy.load("en_core_web_sm")
    df = pd.read_csv(file)
    data: list[Dialog] = []
    for row in df.iloc:
        dialogue = ast.literal_eval(row['Dialogue'])['text']
        dialogue_data: Dialog = []
        for text in dialogue:
            if text['response'] == '$S$':
                continue
            if text['response'] == '$EXIT$':
                break
            role = 1 if text['role'] == 'A' else -1
            doc = nlp(text['response'])
            embeddings_data: list[np.ndarray] = []
            for token in doc:
                if token.lemma_ in embeddings:
                    embeddings_data.append(embeddings[token.lemma_])
            embeddings_data = np.array(embeddings_data)
            dialogue_data.append((role, embeddings_data))
        data.append(dialogue_data)
    with open("features.pkl", "wb") as f:
        pickle.dump(data, f)
    return data

def load_data(file: str = "features.pkl") -> list[Dialog]:
    with open(file, "rb") as f:
        data = pickle.load(f)
    return data

Preprocessing by Yibai

In [ ]:
from torch.nn.utils.rnn import PackedSequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pad_packed_sequence, pack_padded_sequence, unpack_sequence

In [ ]:
def to_inputs_and_labels(data: list[Dialog]) -> tuple[list[list[np.ndarray]], list[list[Literal[-1, 1]]]]:
    '''Split data into inputs and labels

    Args:
        data(list[Dialog]): loaded training data, with non-uniform batch size
        and sequence length.  
                
    :Returns: tuple[inputs, labels] WHERE

        inputs(list[list[np.ndarray]]): input data, with non-uniform batch size
        and sequence length.
        
        labels(list[list[Literal[-1, 1]]]): labels for each sequence, with
        non-uniform batch size and sequence length.
    '''
    inputs = []
    labels = []
    for batch in data:
        inputs_batch = []
        labels_batch = []
        for pair in batch:
            inputs_batch.append(pair[1])
            labels_batch.append(pair[0])
        inputs.append(inputs_batch)
        labels.append(labels_batch)
    return (inputs, labels)

In [ ]:
def is_valid_batch(
    inputs: list[np.ndarray],
    labels: list[Literal[-1, 1]],
    hidden_size: int
) -> bool:
    '''Verify whether the inputs and labels of a batch have valid vector representations.

    Args:
        inputs(list[np.ndarray]): list of all sequences in the batch
        labels(list[Literal[-1, 1]]): list of labels for each sequence in the batch
        hidden_size(int): hidden dimension size
    '''
    # Verify that inputs and labels are non-empty:
    if len(inputs) == 0 or len(labels) == 0:
        return False
    
    # Verify that all sequences in the batch are non-empty:
    for seq in inputs:
        if seq.shape == (0, ):
            return False
    
    # Verify that all tokens in the batch have the expected size of embedding:
    for seq in inputs:
        if seq.shape[1] != hidden_size:
            return False
    
    # Verify that inputs and labels have compatible dimensions:
    if len(inputs) != len(labels):
        return False

    return True

def filter_good_batches(
    inputs: list[list[np.ndarray]],
    labels: list[list[Literal[-1, 1]]],
    hidden_size: int
) -> tuple[list[list[np.ndarray]], list[list[Literal[-1, 1]]]]:
    '''Remove all batches containing invalid data

    Args:
        inputs(list[list[np.ndarray]]): inputs for all batches
        labels(list[list[Literal[-1, 1]]]): labels every sequences
    '''
    inputs_copy = inputs.copy()
    labels_copy = labels.copy()
    num_batch = len(inputs)
    assert(num_batch == len(labels))
    
    pop_count = 0
    for i in range(num_batch):
        if not is_valid_batch(inputs[i], labels[i], hidden_size):
            inputs_copy.pop(i - pop_count)
            labels_copy.pop(i - pop_count)
            pop_count += 1
    
    return (inputs_copy, labels_copy)

In [ ]:
def pad_inputs(inputs: list[list[np.ndarray]]) -> list[torch.Tensor]:
    '''Pad the sequences with tokens, followed by padding batches with sequences,
    to align the dimensions.

    Returns:
        updated_inputs(list(torch.Tensor)): Input in processable format by pytorch
        rnn modules.
    '''
    updated_inputs = []
    for batch in inputs:
        # Batch format: from list[np.ndarray] to list[torch.Tensor]
        batch = [torch.tensor(seq, dtype=torch.float32) for seq in batch]
        
        padded_batch = pad_sequence(batch, padding_side="left", batch_first=True)
        updated_inputs.append(padded_batch)
    
    # padded_inputs = pad_sequence(updated_inputs, batch_first=True)
    # return padded_inputs
    return updated_inputs

In [ ]:
def collate_inputs(inputs: list[list[np.ndarray]]) -> tuple[torch.Tensor, torch.Tensor]:
    '''Pad sequences to global max_seq_len. Then pad batches to max_batch_size.
    Use this function instead of calling nn.utils.rnn.pad_sequence on each batch,
    because each batch has its own max_seq_len,
    and we need to use the global max.

    Args:
        inputs(list[list[np.ndarray]]): All input data
    
    Returns:
        tuple(tuple[torch.Tensor, torch.Tensor]): Input tensor
        in the shape (num_batch, max_batch_size, max_seq_len)
        and seq_lens tensor in the shape (num_batch, max_batch_size).
    '''
    # Flatten seqs across all dialogs and convert type from np.ndarray to torch.Tensor
    all_sequences = [torch.tensor(seq) for dialog in inputs for seq in dialog]

    # Find the global max sequence length
    max_seq_len = max(seq.size(0) for seq in all_sequences)

    # Pad each sequence to max_seq_len
    padded_sequences = [F.pad(seq, (0, 0, 0, max_seq_len - seq.size(0))) for seq in all_sequences]

    # Group padded sequences back into dialog structure
    batch_size = len(inputs)
    max_batch_size = max(len(dialog) for dialog in inputs)

    padded_dialogs = []
    seq_lens = []

    idx = 0
    for dialog in inputs:
        padded_dialogs.append(torch.stack(padded_sequences[idx:idx + len(dialog)]))
        seq_lens.append(torch.tensor([len(seq) for seq in dialog]))
        idx += len(dialog)

    # Pad dialogs with zero sequences to match max_batch_size
    for i in range(batch_size):
        while len(padded_dialogs[i]) < max_batch_size:
            padded_dialogs[i] = torch.cat((padded_dialogs[i], torch.zeros(1, max_seq_len, padded_dialogs[i].size(-1))), dim=0)
            seq_lens[i] = torch.cat((seq_lens[i], torch.tensor([0])))

    return torch.stack(padded_dialogs), torch.stack(seq_lens)

In [ ]:
def pack_wrapper(X: torch.Tensor, seq_lens: torch.Tensor) -> PackedSequence:
    '''Mask the padding in input tensor to improve computational efficiency.
    This is a wrapper to specify reshaping operations involved.

    Returns:
        packed_X(torch.nn.utils.rnn.PackedSequence): packed input tensor
    '''
    flattened_X = X.view(-1, X.size(-2), X.size(-1))
    flattened_seq_lens = seq_lens.view(-1)

    # Filter out zero-length sequences, since pack_padded_sequence() only
    # accepts non-empty sequences.
    non_empty_mask = flattened_seq_lens > 0
    filtered_X = flattened_X[non_empty_mask]
    filtered_seq_lens = flattened_seq_lens[non_empty_mask]
    
    # Pack the sequences
    return pack_padded_sequence(filtered_X, 
                                filtered_seq_lens, 
                                batch_first=True,
                                enforce_sorted=False)

In [ ]:
data = load_data("../features.pkl")
HIDDEN_SIZE = 50

inputs, labels = to_inputs_and_labels(data)
inputs, labels = filter_good_batches(inputs, labels, HIDDEN_SIZE)
X, seq_lens = collate_inputs(inputs)

Simple LSTM Model

In [ ]:
EMBED_SIZE = 50
HIDDEN_SIZE = 32

In [ ]:
class TokenLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(TokenLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True)

    def forward(self, X: torch.Tensor, seq_lens: torch.Tensor):
        packed_X = pack_wrapper(X, seq_lens)
        packed_y, _ = self.lstm(packed_X)
        y, _ = pad_packed_sequence(packed_y, batch_first=True)
        return y

In [ ]:
tokenLSTM = TokenLSTM(50, 3, 1)
seq_embed = tokenLSTM.forward(X, seq_lens)

In [ ]:
class SequenceLSTM(nn.Module):
    def __init__(self, hidden_size, num_classes=2):
        super(SequenceLSTM, self).__init__()
        self.seq_lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, X, seq_lens):
        # Pack padded sequences for efficient processing
        packed_seq = pack_padded_sequence(X, seq_lens, batch_first=True, enforce_sorted=False)

        # Sequence-level LSTM
        packed_output, (h_n, _) = self.seq_lstm(packed_seq)

        # Extract final sequence-level hidden state
        seq_embeddings, _ = pad_packed_sequence(packed_output, batch_first=True)
        logits = self.fc(seq_embeddings[:, -1, :])  # Final sequence state

        return logits

In [ ]:
def train(model, inputs, labels):
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

    output = model(inputs)
    output = nn.utils.rnn.unpack_sequence(output)
    loss = loss_function(output, labels)
    loss.backward()
    optimizer.step()

    return loss.item()